---

## ToolCallLimitMiddleware 옵션별 테스트

``ToolCallLimitMiddleware`` 는 에이전트의 **도구(tool) 호출 횟수**를 추적하고,
한도를 넘으면 ``exit_behavior`` 에 따라 다르게 처리합니다.

- 훅: ``after_model`` (모델이 tool_calls 를 낸 직후 한도 확인)

**참고:** [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

아래 각 섹션은 ``feature/MiddlewareToolCallLimit.py`` 의 ``MiddlewareToolCallLimitAgent`` 로
옵션 하나씩 바꿔 동작을 확인합니다.

**옵션 요약:**

| 옵션 | 기본값 | 역할 |
|:---|:---|:---|
| ``tool_name`` | ``None`` (전역) | 제한 대상 도구. ``None`` 이면 모든 도구(``__all__`` 키로 추적) |
| ``thread_limit`` | ``20`` (전역) / ``5`` (도구별) | **스레드 전체** 허용 도구 호출 수 |
| ``run_limit`` | ``10`` (전역) / ``3`` (도구별) | **단일 invoke** 허용 도구 호출 수 |
| ``exit_behavior`` | ``"continue"`` | 한도 초과 시 동작 (아래 표 참고) |

**``exit_behavior`` 세 가지**

| 값 | 동작 | 결과 메시지 |
|:---|:---|:---|
| ``"continue"`` (기본) | 초과 도구만 **차단** (error ``ToolMessage``), 나머지 실행 계속 | ``Tool call limit exceeded. Do not call '...' again.`` |
| ``"end"`` | **즉시 종료** + 차단 ``ToolMessage`` + 안내 ``AIMessage`` | ``'get_weather' tool call limit reached: run limit exceeded (2/1 calls).`` |
| ``"error"`` | ``ToolCallLimitExceededError`` **예외** 발생 | — |

**``thread_limit`` vs ``run_limit``** (``ModelCallLimitMiddleware`` 와 동일한 개념, 대상만 **도구 호출**)

| 구분 | 범위 |
|:---|:---|
| ``run_limit`` | 한 번의 ``invoke()`` 안에서의 도구 호출 |
| ``thread_limit`` | 같은 ``thread_id`` 의 여러 ``invoke()`` 에 누적 |

이중 미들웨어(전역 + ``get_weather`` 전용)를 쓰면 **같은 호출이 두 카운터에 모두** 잡힙니다.
``messages_contain_blocked_tool_call()`` / ``messages_contain_tool_limit_end()`` 로 확인합니다.

In [1]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

from feature.MiddlewareToolCallLimit import (
    MiddlewareToolCallLimitAgent,
    ToolCallLimitExceededError,
    messages_contain_blocked_tool_call,
    messages_contain_tool_limit_end,
)

### 1. 기본 설정 — 이중 미들웨어, 한도 내 정상 응답

원본 노트북과 동일: 전역 ``thread_limit=20`` / ``run_limit=10`` +
``get_weather`` 전용 ``thread_limit=5`` / ``run_limit=3``.
도구 1회 호출 질문은 **정상 완료**되어야 합니다.

In [2]:
agent_default = MiddlewareToolCallLimitAgent()

result = agent_default.invoke(
    inputs={"messages": [HumanMessage(content="What's the weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "tcl-default"}),
)

assert result is not None
assert not messages_contain_blocked_tool_call(result["messages"]), "한도 내 — 차단 ToolMessage 없어야 함"
print("✓ 기본 이중 미들웨어 — 정상 응답:", result["messages"][-1].content[:120])


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_JMvEcP51A1wRQBOx1S1EGEhQ)
 Call ID: call_JMvEcP51A1wRQBOx1S1EGEhQ
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: ToolCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
__all__:
1
get_weather:
1
__all__:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - 

### 2. ``exit_behavior="continue"`` + ``run_limit=1`` — 초과 도구만 차단

여러 도시 날씨를 한 번에 물으면 모델이 **여러 ``get_weather`` 호출**을 낼 수 있습니다.
``run_limit=1`` 이면 1회만 허용하고, 나머지는 error ``ToolMessage`` 로 차단합니다.
에이전트는 **계속 실행**됩니다 (기본 동작).

In [ ]:
agent_continue = MiddlewareToolCallLimitAgent(
    dual_limiters=False,
    tool_name="get_weather",
    thread_limit=None,
    run_limit=1,
    exit_behavior="continue",
)

result = agent_continue.invoke(
    inputs={
        "messages": [
            HumanMessage(
                content="Use get_weather for Seoul, Tokyo, and Paris. Call each city separately."
            )
        ]
    },
    config=RunnableConfig(configurable={"thread_id": "tcl-continue"}),
)

blocked = messages_contain_blocked_tool_call(result["messages"])
print(f"✓ exit_behavior=continue, run_limit=1 — 차단 ToolMessage 존재: {blocked}")
if blocked:
    err_tools = [
        m.content for m in result["messages"]
        if hasattr(m, "status") and m.status == "error"
    ]
    print("  차단 메시지:", err_tools[0] if err_tools else "(없음)")


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ZB2OLjGQHEOOEi7r6cEmkqjy)
 Call ID: call_ZB2OLjGQHEOOEi7r6cEmkqjy
  Args:
    city: Seoul
  get_weather (call_adAAA1os9hxD9qs5A8YMmNDX)
 Call ID: call_adAAA1os9hxD9qs5A8YMmNDX
  Args:
    city: Tokyo
  get_weather (call_1aDqjFG74l4iHBOMSyOd3NpL)
 Call ID: call_1aDqjFG74l4iHBOMSyOd3NpL
  Args:
    city: Paris

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
3
================================= Tool Message =================================
Name: get_weather

Tool call limit exceeded. Do not call 'get_weather' again.
================================= Tool Message =================================
Name: get_weather

Tool call limit exceeded. Do not call 'get_weather' again.

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - -

### 3. ``exit_behavior="end"`` + ``run_limit=1`` — 즉시 종료

한도 초과 시 ``jump_to: end`` 로 실행을 끝내고, 사용자용 ``AIMessage`` 를 주입합니다.

> **주의:** 모델이 **한 번에 여러 도구**를 병렬 호출하면 ``NotImplementedError`` 가 날 수 있습니다.
> (다른 도구가 pending 상태이기 때문) 단일 ``get_weather`` 호출 질문이 안전합니다.

In [5]:
agent_end = MiddlewareToolCallLimitAgent(
    dual_limiters=False,
    tool_name="get_weather",
    thread_limit=2,
    run_limit=1,
    exit_behavior="end",
)

cfg = RunnableConfig(configurable={"thread_id": "tcl-end"})

# 1회 invoke — 정상 (run_limit=1 이내)
r1 = agent_end.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Seoul only.")]},
    config=cfg,
)
assert not messages_contain_tool_limit_end(r1["messages"]), "1회 — 정상"

# 2회 invoke — thread 누적 후 한도 초과 → end
r2 = agent_end.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Tokyo only.")]},
    config=cfg,
)
ended = messages_contain_tool_limit_end(r2["messages"])
print(f"✓ exit_behavior=end — 2회 invoke 후 종료 메시지: {ended}")
if ended:
    end_msg = next(
        m.content for m in r2["messages"] if messages_contain_tool_limit_end([m])
    )
    print(" ", end_msg)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_BFq2ODlFVj2Z9MrdWluKk5wI)
 Call ID: call_BFq2ODlFVj2Z9MrdWluKk5wI
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai M

### 4. ``exit_behavior="error"`` — ``ToolCallLimitExceededError``

한도 초과 시 예외가 발생합니다. 예외 객체에 ``thread_count`` / ``run_count`` 등이 있습니다.

In [6]:
agent_error = MiddlewareToolCallLimitAgent(
    dual_limiters=False,
    tool_name="get_weather",
    thread_limit=1,
    run_limit=1,
    exit_behavior="error",
)

cfg = RunnableConfig(configurable={"thread_id": "tcl-error"})

agent_error.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Seoul.")]},
    config=cfg,
)

try:
    agent_error.invoke(
        inputs={"messages": [HumanMessage(content="Weather in Tokyo.")]},
        config=cfg,
    )
    raise AssertionError("thread_limit=1 초과 시 ToolCallLimitExceededError 기대")
except ToolCallLimitExceededError as e:
    print(f"✓ exit_behavior=error — {type(e).__name__}: {e}")
    print(f"  thread_count={e.thread_count}, tool_name={e.tool_name}")


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_nRHYaaySQ8G9vIweyv8LFDqD)
 Call ID: call_nRHYaaySQ8G9vIweyv8LFDqD
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai M

### 5. ``tool_name="get_weather"`` — 특정 도구만 제한

``get_weather`` 만 ``run_limit=1`` 로 제한하고, ``get_time`` 은 제한하지 않습니다.
(``tool_name=None`` 이면 모든 도구에 동일 제한)

In [7]:
agent_tool_only = MiddlewareToolCallLimitAgent(
    dual_limiters=False,
    tool_name="get_weather",
    thread_limit=None,
    run_limit=1,
    exit_behavior="continue",
)

result = agent_tool_only.invoke(
    inputs={
        "messages": [
            HumanMessage(
                content="Call get_weather for Seoul AND get_time for Seoul. Both tools."
            )
        ]
    },
    config=RunnableConfig(configurable={"thread_id": "tcl-tool-name"}),
)

blocked = messages_contain_blocked_tool_call(result["messages"])
print(f"✓ tool_name=get_weather — get_weather 한도 초과 차단 여부: {blocked}")
print("  최종 응답:", result["messages"][-1].content[:150])


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_mGZidmq5AGEGWFXnjtba3bMS)
 Call ID: call_mGZidmq5AGEGWFXnjtba3bMS
  Args:
    city: Seoul
  get_time (call_UOL7lpBc7zuI3MrZmyFRPdIt)
 Call ID: call_UOL7lpBc7zuI3MrZmyFRPdIt
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_time

It's noon in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai M

### 6. 이중 미들웨어 — 전역 + ``get_weather`` 전용 (원본 패턴)

``make_dual_tool_limit_middlewares()`` / ``dual_limiters=True`` (기본).
같은 ``get_weather`` 호출이 **전역 카운터**와 **도구별 카운터**에 동시에 잡힙니다.

In [8]:
from feature.MiddlewareToolCallLimit import make_dual_tool_limit_middlewares

dual_mws = make_dual_tool_limit_middlewares(
    global_thread_limit=20,
    global_run_limit=10,
    tool_name="get_weather",
    tool_thread_limit=5,
    tool_run_limit=3,
)
print("미들웨어:", [m.name for m in dual_mws])

agent_dual = MiddlewareToolCallLimitAgent(middlewares=dual_mws)
result = agent_dual.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "tcl-dual"}),
)
print("✓ 이중 미들웨어 — 정상:", result["messages"][-1].content[:100])

미들웨어: ['ToolCallLimitMiddleware', 'ToolCallLimitMiddleware[get_weather]']

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_xoQIi87LxQlGnDaeBSgEsIRL)
 Call ID: call_xoQIi87LxQlGnDaeBSgEsIRL
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: ToolCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
__all__:
1
get_weather:
1
__all__:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is s

In [15]:
result

{'messages': [HumanMessage(content='Weather in Seoul?', additional_kwargs={}, response_metadata={}, id='7ad52480-d442-46d1-8917-f1707885f1ce'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 73, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_032c106278', 'id': 'chatcmpl-DoODKbExAWpC2FTzjw0NPLResFk7J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea605-aa10-75b2-a8a6-b5dac348c8e1-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seoul'}, 'id': 'call_xoQIi87LxQlGnDaeBSgEsIRL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 15

### 7. ``thread_limit=1`` — 같은 스레드 2번째 invoke 차단

``thread_tool_call_count`` 가 스레드에 누적됩니다.
``checkpointer`` + 같은 ``thread_id`` 로 2번 invoke 하면 2번째에서 차단됩니다.

In [14]:
agent_thread = MiddlewareToolCallLimitAgent(
    tool_name="get_weather",
    thread_limit=2,
    run_limit=None,
    exit_behavior="continue",
)

cfg = RunnableConfig(configurable={"thread_id": "tcl-thread12"})

r1 = agent_thread.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Seoul.")]},
    config=cfg,
)
assert not messages_contain_blocked_tool_call(r1["messages"]), "1회 — 정상"

r2 = agent_thread.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Seoul.")]},
    config=cfg,
)
assert not messages_contain_blocked_tool_call(r2["messages"]), "2회 — 정상"

r3 = agent_thread.invoke(
    inputs={"messages": [HumanMessage(content="Weather in Tokyo.")]},
    config=cfg,
)
blocked = messages_contain_blocked_tool_call(r3["messages"])
print(f"✓ thread_limit=1 — 2회 invoke 후 차단: {blocked}")


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_79toVEsexdD7F773x6Uh3yKw)
 Call ID: call_79toVEsexdD7F773x6Uh3yKw
  Args:
    city: Seoul

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
get_weather:
1

🔄 Node: ToolCallLimitMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
get_weather:
1
__all__:
1
get_weather:
1
__all__:
1

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - 

### 8. ``run_limit`` 만 설정 — invoke마다 run 카운트 리셋

``thread_limit=None`` 이면 스레드 누적 없이 **invoke 단위**만 제한합니다.
단순 질문(도구 1회)은 invoke 를 반복해도 매번 정상입니다.

In [11]:
agent_run_only = MiddlewareToolCallLimitAgent(
    dual_limiters=False,
    tool_name="get_weather",
    thread_limit=None,
    run_limit=1,
    exit_behavior="continue",
)

cfg = RunnableConfig(configurable={"thread_id": "tcl-run-only"})

for i in range(2):
    r = agent_run_only.invoke(
        inputs={"messages": [HumanMessage(content=f"Weather in city {i+1}.")]},
        config=cfg,
    )
    blocked = messages_contain_blocked_tool_call(r["messages"])
    print(f"  invoke {i+1}: {'차단 있음' if blocked else '정상'}")

print("✓ run_limit=1, thread_limit=None — invoke마다 리셋, 단일 호출은 매번 정상")


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Could you please provide me with the name of the city for which you would like to know the weather?

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
  invoke 1: 정상

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

I need the specific name of the city in order to provide you with the weather information. Could you please specify the city?

🔄 Node: ToolCallLimitMiddleware[get_weather].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
  invoke 2: 정상
✓ run_limit=1, thread_limit=None — invoke마다 리셋, 단일 호출은 매번 정상


### 9. (검증) 잘못된 설정 — ``ValueError``

- ``thread_limit`` 와 ``run_limit`` 를 **둘 다** ``None``
- ``run_limit > thread_limit``

In [12]:
from feature.MiddlewareToolCallLimit import make_tool_call_limit_middleware

cases = [
    ("둘 다 None", {"thread_limit": None, "run_limit": None}),
    ("run > thread", {"thread_limit": 2, "run_limit": 5}),
]

for label, kwargs in cases:
    try:
        make_tool_call_limit_middleware(**kwargs)
        print(f"⊘ {label}: ValueError 기대했으나 통과")
    except ValueError as e:
        print(f"✓ {label} — ValueError: {e}")

✓ 둘 다 None — ValueError: At least one limit must be specified (thread_limit or run_limit)
✓ run > thread — ValueError: run_limit (5) cannot exceed thread_limit (2). The run limit should be less than or equal to the thread limit.
